# Limpieza y homologación

Pipeline reproducible que produce los dos datasets finales en `data/processed/`: `dataset_final.parquet` (partido) y `historico_galan_limpio.parquet` (Galán). Lee solo raw versionados + shapefiles, con rutas relativas.

In [1]:
import os
os.environ["SHAPE_ENCODING"] = "UTF-8"  # DBF en UTF-8; sin esto OGR asume cp1252 y corrompe vocales
import unicodedata
import re
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

# El cuaderno corre desde notebooks/ o desde la raíz: anclar ROOT al repo
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw" / "2021_2026"
GEO = ROOT / "data" / "raw" / "geodata"
PROC = ROOT / "data" / "processed"
OUT_PARTIDO = PROC / "nuevo_liberalismo.parquet"
OUT_GALAN = PROC / "carlos_fernando_galan.parquet"

def normalizar_texto(x):
    """Minúsculas, sin tildes, sin caracteres de control, espacios colapsados. NaN se conserva."""
    if pd.isna(x):
        return x
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c)
                and unicodedata.category(c) != "Cc")
    return " ".join(x.split())

def norm_key(x):
    """Clave para joins con shapefiles: solo [a-z0-9 ]."""
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

def leer_csv(p):
    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
        try:
            return pd.read_csv(p, low_memory=False, encoding=enc)
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f"Sin encoding válido: {p}")

def buscar_codigo(ref, *fragmentos):
    for k, v in ref.items():
        if all(f in k for f in fragmentos):
            return v
    raise KeyError(f"Sin match para {fragmentos}")

def norm_cols_texto(df):
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].map(normalizar_texto)
    return df

print("ROOT:", ROOT)
print("raw existe:", RAW.exists(), "| geodata existe:", GEO.exists())


ROOT: C:\Users\qluis\Documents\Proyectos\Analítica electoral
raw existe: True | geodata existe: True


## 0. Diccionarios de referencia

Reglas de homologación de texto (departamento, municipio, corporación, candidato). Se aplican tal cual, sin lógica intermedia.

In [2]:
reemplazos_departamento = {
    'bogota dc': 'bogota',
    'santafe de bogota dc': 'bogota',
    'bogota d.c.': 'bogota',
    'valle': 'valle del cauca',
    'nariã‘o': 'narino',
    'norte de san': 'norte de santander',
    'san andres': 'san andres, providencia y santa catalina',
    'comisaria del vichada': 'vichada'# o 'desconocido' si prefieres
}

reemplazos_municipio = {
    # Caracteres extraños / ñ
    'brice?o': 'briceno',
    'briceã‘o': 'briceno',
    'ca?asgordas': 'canasgordas',
    'cove?as': 'covenas',
    'coveã‘as': 'covenas',
    'el pe?ol': 'el penol',
    'el peã‘ol': 'el penol',
    'el pe?on': 'el penon',
    'el peã‘on': 'el penon',
    'el pi?on': 'el pinon',
    'el piã‘on': 'el pinon',
    'la monta?ita': 'la montanita',
    'la montaã‘ita': 'la montanita',
    'la pe?a': 'la pena',
    'la peã‘a': 'la pena',
    'mo?itos': 'monitos',
    'moã‘itos': 'monitos',
    'nari?o': 'narino',
    'nariã‘o': 'narino',
    'oca?a': 'ocana',
    'ocaã‘a': 'ocana',
    'piji?o del carmen': 'pijino del carmen',
    'pijiã‘o del carmen': 'pijino del carmen',
    'puerto carre?o': 'puerto carreno',
    'puerto carreã‘o': 'puerto carreno',
    'puerto nari?o': 'puerto narino',
    'puerto nariã‘o': 'puerto narino',
    'salda?a': 'saldana',
    'saldaã‘a': 'saldana',
    'san jose de la monta?a': 'san jose de la montana',

    # Bogotá
    'bogota d e': 'bogota',
    'bogota dc': 'bogota',
    'bogota. d.c.': 'bogota',
    'santafe de bogota dc': 'bogota',

    # Paréntesis / cabeceras equivalentes
    'alban (san jose)': 'alban',
    'alto baudo (pie de pato)': 'alto baudo',
    'aquitania (puebloviejo)': 'aquitania',
    'arboleda (berruecos)': 'arboleda',
    'ariguani (el dificil)': 'ariguani',
    'armero (guayabal)': 'armero',
    'atrato (yuto)': 'atrato',
    'bahia solano (mutis)': 'bahia solano',
    'bajo baudo (pizarro)': 'bajo baudo',
    'bojaya (bellavista)': 'bojaya',
    'buenos aires (pacoa)': 'buenos aires',
    'calima (darien)': 'calima',
    'colon (genova)': 'colon',
    'coloso (ricaurte)': 'coloso',
    'cotorra (bongo)': 'cotorra',
    'cuaspud (carlosama)': 'cuaspud',
    'francisco pizarro (salahonda)': 'francisco pizarro',
    'galeras (nueva granada)': 'galeras',
    'la apartada (frontera)': 'la apartada',
    'la argentina (plata vieja)': 'la argentina',
    'lopez (micay)': 'lopez',
    'los andes (sotomayor)': 'los andes',
    'magui (payan)': 'magui',
    'mallama (piedrancha)': 'mallama',
    'medio atrato (bete)': 'medio atrato',
    'medio baudo (puerto meluk)': 'medio baudo',
    'paez (belalcazar)': 'paez',
    'paratebueno (la naguaya)': 'paratebueno',
    'patia (el bordo)': 'patia',
    'paz de ariporo (moreno)': 'paz de ariporo',
    'purace (coconuco)': 'purace',
    'roberto payan (san jose)': 'roberto payan',
    'san juan de betulia (betulia)': 'san juan de betulia',
    'san miguel (la dorada)': 'san miguel',
    'santa barbara (iscuande)': 'santa barbara',
    'santacruz (guachaves)': 'santacruz',
    'sotara (paispamba)': 'sotara',
    'tesalia (carnicerias)': 'tesalia',
    'zona bananera (sevilla)': 'zona bananera',

    # Duplicados o variantes
    'arbeleez': 'arbelaez',
    'calara': 'calarca',
    'don matias': 'donmatias',
    'cuaspud carlosama': 'cuaspud',
    'guadalajara de buga': 'buga',
    'guican de la sierra': 'guican',
    'manaure balcon del cesar (mana': 'manaure balcon del cesar',
    'palmas socorro': 'palmas del socorro',
    'pueblo rico': 'pueblorrico',
    'puerto nare (la magdalena)': 'puerto nare',
    'puerto nare-la magdalena': 'puerto nare',
    'san carlos guaroa': 'san carlos de guaroa',
    'san jose del fragua': 'san jose de fragua',
    'san pablo borbur': 'san pablo de borbur',
    'santacruz': 'santa cruz',
    'santiago de tolu': 'tolu',
    'tiquisio (pto. rico)': 'tiquisio',
    'villa de leyva': 'villa de leiva',
    'vistahermosa': 'vista hermosa',
    'yondo-casabe': 'yondo',

    # Casos truncados / incompletos
    'el canton del san pablo (man.': 'el canton del san pablo',
    'union panamericana (las animas': 'union panamericana'
}

reemplazos_corporacion = {
    'asamblea': 'asamblea departamental',
    'asamblea departamental': 'asamblea departamental',
    'concejo municipal': 'concejo',
    'concejo': 'concejo',
    'camara': 'camara',
    'camara de representantes': 'camara',
    'jal': 'jal',
    'senado': 'senado',
    'alcaldia municipal': 'alcaldia',
    'alcalde': 'alcaldia',
    'presidencia': 'presidencia',
    'gobernador': 'gobernacion',
    'consultas': 'consultas'
}

valores_a_reemplazar = [
    'nuevo liberalismo- agrupacion politica en marcha',
    'nuevo liberalismo - conservador -colombia just...',
    'partido de la u-nuevo liberalismo-colombia jus...',
    'nuevo liberalismo-nueva fuerza democratica',
    'u, nuevo liberalismo, mira',
    'coalicion nuevo liberalismo en marcha',
    'nuevo liberalismo - cambio radical',
    'nuevo liberalismo y mira al concejo',
    'coalicion nuevo liberalismo - dignidad y compr...',
    'coalicion nuevo liberalismo y mira',
    'colombia renaciente - nuevo liberalismo',
    'nuevo liberalismo en marcha',
    'nuevo liberalismo - mira',
    'dignidad & compromiso y nuevo liberalismo',
    'alianza verde nuevo liberalismo'
]
reemplazos_df1_candidato = {
    'movimiento de izquierda nuevo liberal 82': 'lista',
    'partido nuevo liberalismo larrarte rodriguez': 'larrarte rodriguez',
    'movimiento nuevo liberalismo': 'lista',
    'partido nuevo liberalismo quijano caballero': 'quijano caballero',
    'partido nuevo liberalismo': 'lista',
}

reemplazos_candidato_final = {
    'lista - mira': 'lista',
    'nuevo liberalismo - conservador -colombia justa li bres': 'lista',
    'partido nuevo liberalismo': 'lista',
}
valores_df4_lista = {
    'cr-nuevo liberalismo',
    'partido nuevo liberalismo',
    'centro democratico- nuevo liberalismo-mira',
    'partido de la u, mira, nuevo liberalismo'
}


## 1. Partido Nuevo Liberalismo

Carga de los 4 raw (`80_88`, `22_congreso`, `23_territoriales`, `26_congreso`) y mapeo al esquema canónico de 9 columnas.

In [3]:
# ---------- 1. PARTIDO ----------
df1 = leer_csv(RAW / "80_88_nuevo_liberalismo.csv")
print("df1 raw:", df1.shape)
print("nombre_completo con 'sin_dato':", df1["nombre_completo"].fillna("").str.contains("sin_dato").sum())
df1 = df1.drop(columns=["id_electoral", "circunscripcion", "codigo_partido",
                        "primer_apellido", "segundo_apellido", "nombres"], errors="ignore")
df1 = norm_cols_texto(df1).fillna("sin_dato")
df1["nombre_completo"] = df1["nombre_completo"].replace(reemplazos_df1_candidato)
df1 = df1.rename(columns={"nombre_completo": "candidato", "tipo_eleccion": "corporacion"})
df1["partido"] = "nuevo liberalismo"

df2 = leer_csv(RAW / "22_congreso_nuevo_liberalismo.csv")
df2 = df2[["Código Departamento", "Nombre Departamento", "Código Municipio",
           "Nombre Municipio", "Nombre Corporación", "Nombre Candidato", "Total Votos"]]
df2 = df2.rename(columns={"Código Departamento": "coddpto", "Nombre Departamento": "departamento",
                          "Código Municipio": "codmpio", "Nombre Municipio": "municipio",
                          "Nombre Corporación": "corporacion", "Nombre Candidato": "candidato",
                          "Total Votos": "votos"})
df2["partido"] = "nuevo liberalismo"
df2["ano"] = 2022
df2 = norm_cols_texto(df2)
df2["candidato"] = df2["candidato"].str.replace("partido nuevo liberalismo", "lista", regex=False)

df3 = leer_csv(RAW / "23_territoriales_nuevo_liberalismo.csv")
df3 = df3[["Código Departamento", "Nombre Departamento", "Código Municipio",
           "Nombre Municipio", "Nombre Corporación", "Nombre Candidato",
           "Nombre Partido", "Total Votos"]]
df3 = df3.rename(columns={"Código Departamento": "coddpto", "Nombre Departamento": "departamento",
                          "Código Municipio": "codmpio", "Nombre Municipio": "municipio",
                          "Nombre Corporación": "corporacion", "Nombre Candidato": "candidato",
                          "Nombre Partido": "partido", "Total Votos": "votos"})
df3 = norm_cols_texto(df3)
df3["candidato"] = df3["candidato"].str.replace("partido nuevo liberalismo", "lista", regex=False)
df3["ano"] = 2023
df3.loc[df3["candidato"].isin(valores_a_reemplazar), "candidato"] = "lista"

df4 = leer_csv(RAW / "26_congreso_nuevo_liberalismo.csv")
df4 = df4[["DEP", "DEPNOMBRE", "MUN", "MUNNOMBRE", "CORNOMBRE", "CANNOMBRE", "PARNOMBRE", "VOTOS"]]
df4 = df4.rename(columns={"DEP": "coddpto", "DEPNOMBRE": "departamento", "MUN": "codmpio",
                          "MUNNOMBRE": "municipio", "CORNOMBRE": "corporacion",
                          "CANNOMBRE": "candidato", "PARNOMBRE": "partido", "VOTOS": "votos"})
df4 = norm_cols_texto(df4)
df4["ano"] = 2026
df4.loc[df4["candidato"].isin(valores_df4_lista), "candidato"] = "lista"
mask_cd = df4["candidato"].str.contains("centro democr", case=False, na=False)
df4.loc[mask_cd, "candidato"] = "lista"

maestro = pd.concat([df1, df2, df3, df4], ignore_index=True)
maestro["votos"] = pd.to_numeric(maestro["votos"], errors="coerce").fillna(0).round().astype("int64")
for c in ["coddpto", "codmpio", "departamento", "municipio"]:
    maestro[c] = maestro[c].astype(str).str.strip()
print("maestro:", maestro.shape)

# BOGOTA se unifica DESPUES del diccionario de municipios
# (fuentes traen 'bogota dc', 'bogota. d.c.', etc.)
maestro["departamento"] = maestro["departamento"].replace(reemplazos_departamento)
# Fuente 22/23 con caracter perdido irrecuperable: todo lo que contenga 'nari' es Nariño
# (verificado: únicos valores son 'narino' y la variante corrupta)
maestro.loc[maestro["departamento"].str.contains("nari", na=False), "departamento"] = "narino"
maestro.loc[maestro["departamento"] == "archipielago de san andres providencia y santa catalina",
            "departamento"] = "san andres, providencia y santa catalina"
maestro["municipio"] = maestro["municipio"].replace(reemplazos_municipio)
n_bog = ((maestro["municipio"] == "bogota") & (maestro["departamento"] != "bogota")).sum()
maestro.loc[maestro["municipio"] == "bogota", "departamento"] = "bogota"
print("filas bogota unificadas a departamento=bogota:", n_bog)
maestro["corporacion"] = maestro["corporacion"].replace(reemplazos_corporacion)
maestro["candidato"] = maestro["candidato"].replace(reemplazos_candidato_final)
mask_lista = maestro["candidato"].str.contains(r"^lista\b", case=False, na=False)
maestro.loc[mask_lista, "candidato"] = "lista"
maestro["clave_territorial"] = maestro["coddpto"] + "-" + maestro["codmpio"]



df1 raw: (14142, 14)
nombre_completo con 'sin_dato': 0


maestro: (1092364, 9)


filas bogota unificadas a departamento=bogota: 16


## 2. Códigos canónicos y clave territorial

Incluye la unificación de Bogotá (todo `municipio == bogota` → `departamento == bogota`).

In [4]:
# códigos canónicos (prioridad año reciente)
df = maestro.copy()
def norm_key(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()
df["ano"] = pd.to_numeric(df["ano"], errors="coerce")
df["departamento_norm"] = df["departamento"].map(norm_key)
df["municipio_norm"] = df["municipio"].map(norm_key)
prioridad = {2026: 6, 2023: 5, 2022: 4, 1988: 3, 1986: 2, 1982: 1}
valid = df[df["departamento_norm"].notna() & df["municipio_norm"].notna()].copy()
valid["prioridad"] = valid["ano"].map(prioridad).fillna(0).astype(int)
resumen = valid.groupby(["departamento_norm", "municipio_norm", "coddpto", "codmpio", "ano"],
                        dropna=False).size().reset_index(name="registros")
resumen["prioridad"] = resumen["ano"].map(prioridad).fillna(0).astype(int)
canon = (resumen.sort_values(["departamento_norm", "municipio_norm", "prioridad", "registros"],
                             ascending=[True, True, False, False])
         .groupby(["departamento_norm", "municipio_norm"], as_index=False).first())
canon = canon.rename(columns={"coddpto": "coddpto_canon", "codmpio": "codmpio_canon"})
df2m = df.merge(canon[["departamento_norm", "municipio_norm", "coddpto_canon", "codmpio_canon"]],
                on=["departamento_norm", "municipio_norm"], how="left")
df2m["coddpto"] = df2m["coddpto_canon"].combine_first(df2m["coddpto"])
df2m["codmpio"] = df2m["codmpio_canon"].combine_first(df2m["codmpio"])
df2m["clave_territorial"] = df2m["coddpto"].astype(str).str.strip() + "-" + df2m["codmpio"].astype(str).str.strip()
tmp = df2m[["coddpto", "codmpio", "departamento", "municipio",
            "departamento_norm", "municipio_norm", "ano"]].copy()
tmp["prioridad"] = tmp["ano"].map(prioridad).fillna(0).astype(int)
pref = (tmp.groupby(["coddpto", "codmpio", "departamento", "municipio",
                     "departamento_norm", "municipio_norm", "prioridad"]).size()
        .reset_index(name="registros")
        .sort_values(["coddpto", "codmpio", "prioridad", "registros"],
                     ascending=[True, True, False, False])
        .groupby(["coddpto", "codmpio"], as_index=False).first())
pref = pref.rename(columns={"departamento": "departamento_canon", "municipio": "municipio_canon"})
df2m = df2m.merge(pref[["coddpto", "codmpio", "departamento_canon", "municipio_canon"]],
                  on=["coddpto", "codmpio"], how="left")
df2m["departamento"] = df2m["departamento_canon"].combine_first(df2m["departamento"])
df2m["municipio"] = df2m["municipio_canon"].combine_first(df2m["municipio"])

# cod_dpto_geo via join por NOMBRE con shape departamentos (cp1252 decodifica Ñ)
deptos = gpd.read_file(str(GEO / "departamentos" / "MGN_ADM_DPTO_POLITICO.shp"))
ref_dpto = {norm_key(n): c for n, c in zip(deptos["dpto_cnmbr"], deptos["dpto_ccdgo"])}
# alias: bogota (shape trae 'BOGOTA, D.C.') y san andres (nombre largo con
# caracteres perdidos en el DBF). Búsqueda por fragmentos, no por texto exacto.
def buscar_codigo(ref, *fragmentos):
    for k, v in ref.items():
        if all(f in k for f in fragmentos):
            return v
    raise KeyError(f"Sin match para {fragmentos}")
ref_dpto["bogota"] = buscar_codigo(ref_dpto, "bogot")
ref_dpto["san andres providencia y santa catalina"] = buscar_codigo(ref_dpto, "catalina")
df2m["cod_dpto_geo"] = df2m["departamento"].map(norm_key).map(ref_dpto)
sin_geo = df2m[df2m["cod_dpto_geo"].isna()]
print("filas sin cod_dpto_geo:", len(sin_geo))
print(sin_geo.groupby("departamento").size().to_string())

# ---------- unificación partido / coaliciones (35 -> 19) ----------
REEMPLAZOS_PARTIDO = {
    "partido nuevo liberalismo": "nuevo liberalismo",
    "coalicion nuevo liberalismo en marcha": "nuevo liberalismo en marcha",
    "nuevo liberalismo y en marcha": "nuevo liberalismo en marcha",
    "nuevo liberalismo- agrupacion politica en marcha": "nuevo liberalismo en marcha",
    "partido nuevo liberalismo-agrupacion politica en marcha": "nuevo liberalismo en marcha",
    "centro democratico- nuevo liberalismo-mira": "centro democratico - nuevo liberalismo - mira",
    "cr-nuevo liberalismo": "cambio radical - nuevo liberalismo",
    "nuevo liberalismo - cambio radical": "cambio radical - nuevo liberalismo",
    "partido cambio radical - nuevo liberalismo": "cambio radical - nuevo liberalismo",
    "alianza verde nuevo liberalismo": "alianza verde - nuevo liberalismo",
    "partido nuevo liberalismo-partido alianza verde": "alianza verde - nuevo liberalismo",
    "nuevo liberalismo-alianza verde": "alianza verde - nuevo liberalismo",
    "partido nuevo liberalismo - mira": "nuevo liberalismo - mira",
    "nuevo liberalismo y mira al concejo": "nuevo liberalismo - mira",
    "nuevo liberalismo - mira": "nuevo liberalismo - mira",
    "coalicion nuevo liberalismo y mira": "nuevo liberalismo - mira",
    "u, nuevo liberalismo, mira": "partido de la u - nuevo liberalismo - mira",
    "partido de la u, mira, nuevo liberalismo": "partido de la u - nuevo liberalismo - mira",
    "coalicion nuevo liberalismo - dignidad y compromiso": "nuevo liberalismo - dignidad y compromiso",
    "dignidad & compromiso y nuevo liberalismo": "nuevo liberalismo - dignidad y compromiso",
    "acuerdo de coalicion partidos de la u y nuevo liberalismo": "partido de la u - nuevo liberalismo",
    "el partido de la union por la gente- partido de la u y el partido nuevo liberalismo": "partido de la u - nuevo liberalismo",
    "partido de la u-nuevo liberalismo-colombia justa libres": "partido de la u - nuevo liberalismo - colombia justa libres",
    "nuevo liberalismo - conservador -colombia justa libres": "nuevo liberalismo - conservador - colombia justa libres",
    "la u-nuevo liberalismo-justa libres-fuerza de la paz": "partido de la u - nuevo liberalismo - justa libres - fuerza de la paz",
    "partido de la u-nuevo liberalismo-movimiento aico": "partido de la u - nuevo liberalismo - movimiento aico",
    "partido de la u-cambio radical-nuevo liberalismo": "partido de la u - cambio radical - nuevo liberalismo",
    "partido de la u-cambio radical-asi-nuevo liberalismo": "partido de la u - cambio radical - asi - nuevo liberalismo",
    "coalicion la u, radical, conservador, nuevo liberalismo": "partido de la u - cambio radical - conservador - nuevo liberalismo",
    "coalicion partido conservador - nuevo liberalismo": "partido conservador - nuevo liberalismo",
}
df2m["partido"] = df2m["partido"].replace(REEMPLAZOS_PARTIDO)
print("partido únicos:", df2m["partido"].nunique(), "(esperado 20)")

COLS = ["ano", "corporacion", "coddpto", "departamento", "codmpio", "municipio",
        "votos", "candidato", "partido", "clave_territorial",
        "coddpto_canon", "codmpio_canon", "cod_dpto_geo"]
final = df2m[COLS].copy()
final.to_parquet(OUT_PARTIDO, index=False)
print("PARTIDO guardado:", final.shape)



filas sin cod_dpto_geo: 8648
departamento
consulados    8646
sin_dato         2


partido únicos: 20 (esperado 20)


PARTIDO guardado: (1092364, 13)


## 3. Galán: IDs de localidad y UPZ

Join por nombre normalizado contra las capas. `geo_flag` marca inconsistencias sin eliminar filas.

In [5]:
# ---------- 2. GALAN ----------
gal = leer_csv(str(ROOT / "data" / "raw" / "historico_carlos_fernando_galan.csv"))
gal.columns = [normalizar_texto(c).replace(" ", "_") for c in gal.columns]
gal = gal.rename(columns={"a_o": "ano", "a?o": "ano"})
if "ano" not in gal.columns:
    gal = gal.rename(columns={gal.columns[0]: "ano"})
print("galan cols:", list(gal.columns), gal.shape)
for c in ["localidad", "upz"]:
    gal[c] = gal[c].map(normalizar_texto)
gal["votacion"] = pd.to_numeric(gal["votacion"], errors="coerce").fillna(0).round().astype("int64")
gal["ano"] = pd.to_numeric(gal["ano"], errors="coerce").astype("Int64")

loc = gpd.read_file(str(GEO / "localidades" / "Loca.shp"))
upz = gpd.read_file(str(GEO / "upz" / "upz-bogota.shp"))
ref_loc = {norm_key(n): str(c).zfill(2) for n, c in zip(loc["LocNombre"], loc["LocCodigo"])}
ref_loc["la candelaria"] = "17"  # shape: CANDELARIA
# UPZ: codigo_upz se repite (60 en locas 5 y 18) -> desambiguar por localidad de la fila
from collections import defaultdict
ref_upz_multi = defaultdict(list)
for n, c, l in zip(upz["nombre"], upz["codigo_upz"], upz["codigo_loca"]):
    ref_upz_multi[norm_key(n)].append((str(c), str(l).zfill(2)))
gal["cod_localidad"] = gal["localidad"].map(norm_key).map(ref_loc)
gal.loc[gal["localidad"] == "nacional", "cod_localidad"] = pd.NA

def mapear_upz(row):
    k = norm_key(row["upz"])
    if pd.isna(k):
        return pd.Series([pd.NA, pd.NA])
    cands = ref_upz_multi.get(k, [])
    if not cands:
        return pd.Series([pd.NA, pd.NA])
    for c, l in cands:
        if pd.notna(row["cod_localidad"]) and l == row["cod_localidad"]:
            return pd.Series([c, l])
    return pd.Series(cands[0])

gal[["cod_upz", "cod_loca_upz"]] = gal.apply(mapear_upz, axis=1)

def flag(r):
    if r["localidad"] == "nacional":
        return "nacional_sin_codigo"
    if pd.isna(r["cod_upz"]):
        return "upz_sin_codigo" if pd.notna(r["upz"]) else "upz_nula"
    if pd.isna(r["cod_localidad"]):
        return "localidad_sin_codigo"
    return "ok" if r["cod_localidad"] == r["cod_loca_upz"] else "inconsistente_loc_upz"
gal["geo_flag"] = gal.apply(flag, axis=1)
print("geo_flag:", gal["geo_flag"].value_counts(dropna=False).to_dict())

GALAN_COLS = ["ano", "corporacion", "lugar", "votacion", "localidad", "cod_localidad",
              "upz", "cod_upz", "longitud", "latitud", "geo_flag"]
galan = gal[GALAN_COLS].copy()
galan.to_parquet(OUT_GALAN, index=False)
print("GALAN guardado:", galan.shape)



galan cols: ['ano', 'corporacion', 'lugar', 'votacion', 'localidad', 'longitud', 'latitud', 'geometry', 'upz'] (3179, 9)


geo_flag: {'ok': 2990, 'inconsistente_loc_upz': 105, 'upz_nula': 74, 'nacional_sin_codigo': 10}
GALAN guardado: (3179, 11)


## 4. Validaciones

Totales esperados: 1.092.364 filas partido; 1.059 filas / 745.738 votos en presidencia 1982 (participación presidencial de Luis Carlos Galán, verificada contra el raw). Nota: en 1986 Galán no fue candidato presidencial en las fuentes (candidatos: Barco, Gómez, Pardo); su participación de ese año fue al Senado y la fuente la registra con votos en lista.

In [6]:
# ---------- validaciones ----------
print("\n== VALIDACIONES PARTIDO ==")
print("total filas:", len(final), "(esperado 1092364)")
p82 = final[(final["ano"] == 1982) & (final["corporacion"] == "presidencia")]
print("1982 presidencia:", len(p82), "votos:", p82["votos"].sum(), "(esperado 1059 / 745738)")
print("bogota-cundinamarca restantes:", ((final["municipio"]=="bogota")&(final["departamento"]!="bogota")).sum(), "(esperado 0)")
print("anos:", sorted(final["ano"].dropna().unique().tolist()))



== VALIDACIONES PARTIDO ==
total filas: 1092364 (esperado 1092364)
1982 presidencia: 1059 votos: 745738 (esperado 1059 / 745738)
bogota-cundinamarca restantes: 0 (esperado 0)
anos: [1982, 1986, 1988, 2022, 2023, 2026]
